In [ ]:
"""
Author: Sophie A. Liu
Date: 06/08/2026 1:53pm
Purpose: isolating local expression activity around each immunofluorescent labeled cell
"""

In [1]:
# importing necessary libraries
import pandas as pd
import numpy as np
from tqdm import tqdm 
import os

In [2]:
# working directory
os.chdir("i:/Hu Lab/Sophie/1. Cell death/visium image manual spot selection/20260413_final_merge/data")

In [4]:
# V = pd.read_csv("0525_NMF_iso.csv")
V = pd.read_csv("0528_NMF_pd1.csv")

# S = pd.read_csv("iso7_coords_clean.csv") # spots from IF
S = pd.read_csv("pd1-9_coords_final.csv")  # spots from IF

In [5]:
# setting parameters/ initializing things
v_coords = V[["x", "y"]].to_numpy()
s_coords = S[["x", "y"]].dropna().to_numpy()

gene_cols = V.columns[3:8]

radius = 40            # balancing capturing enough cells but account for sparsity. 
                       # cell ~ 8 microns. study simplifies to 2D ignoring z-axis.

In [ ]:
print(gene_cols)

In [6]:
from scipy.spatial import cKDTree

In [7]:
def inputs(S, V, gene_cols, s_coords, v_coords):

    # KD-trees
    s_tree = cKDTree(s_coords)
    v_tree = cKDTree(v_coords)

    # encoding cell types as integers leads to faster processing
    type_map = {
        "tdtomato": 0,
        "gc3ai": 1,
        "cd8": 2,
        "lectin": 3
    }
    S_cells = np.array([type_map.get(x, -1) for x in S["cell_type"].values])

    # extracting gene matrix :)
    V_genes = V[gene_cols].to_numpy()

    return s_tree, v_tree, S_cells, V_genes

In [8]:
# counts of each cell type in the neighborhood of a IF-labeled cell, as well as some derived metrics.
def counts_in_radius(center, s_tree, S_cells, radius):

    idx = s_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        counts = np.zeros(4)   # for all four types, if nothing then set 0. Loops through all neighborhoods
    else:
        types = S_cells[idx]
        counts = np.bincount(types[types >= 0], minlength=4) # our result: counts = [n_tdtomato, n_gc3ai, n_cd8, n_lectin]

    n_alive, n_dying, n_immune, n_endothelial = counts       # renaming the channels to what cell type they represent

    # calculating later metrics so I don't have to do it downstream
    tumor = n_alive + n_dying
    total = tumor + n_immune + n_endothelial
    prop_dying = n_dying / tumor                                    # will return some NaN but removed downstream
    exist_dying = 1 if prop_dying > 0 else 0                        # binarizing proportion dying
    eff_cont = prop_dying/ n_immune                                 # dying per cd8, ie. are some T-cells better at killing?
    eff_disc = n_dying / n_immune                   

    return counts, tumor, total, prop_dying, exist_dying, eff_cont, eff_disc

In [9]:
# mean expression of each gene in the neighborhood of a IF-labeled cell, which may be a better representation of gene expression than sums.
# limitation is ignoring density effects. good starting point for now
# mean expression of each gene in the neighborhood of a IF-labeled cell, which may be a better representation of gene expression than sums.
def get_gene_means(center, v_tree, V_genes, radius):
    idx = v_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        return np.zeros(V_genes.shape[1])

    return V_genes[idx].mean(axis=0)

In [10]:
def append_row(center, s_tree, v_tree, S_types, V_genes, radius):

    counts, tumor, total, prop_dying, exist_dying, eff_cont, eff_disc = counts_in_radius(
        center, s_tree, S_types, radius
    )

    gene_means = get_gene_means(
        center, v_tree, V_genes, radius
    )

    row = np.concatenate([
        np.array([center[0], center[1]]),
        counts,
        np.array([tumor, total, prop_dying, exist_dying, eff_cont, eff_disc]),
        gene_means
    ])

    return row

In [11]:
# function for final assembly/joining of neighborhoodresults. vectorizing is quicker but less intuitive. 
def compute_neighborhoods(
    S, V, s_coords, v_coords, gene_cols, radius):

    s_tree, v_tree, S_types, V_genes = inputs(
        S, V, gene_cols, s_coords, v_coords
    )

    n_centers = len(s_coords)
    n_genes = V_genes.shape[1]

    results = np.zeros((n_centers, 12 + n_genes))

    for i, center in enumerate(tqdm(s_coords, desc="Processing")):
        results[i] = append_row(
            center, s_tree, v_tree, S_types, V_genes, radius
        )

    columns = (
        ["cx", "cy",
         "n_alive", "n_dying", "n_immune", "n_lectin",
         "tumor", "all", "prop_dying", "exist_dying", "eff_cont", "eff_disc"]
        + list(gene_cols)
    )

    return pd.DataFrame(results, columns=columns)

In [12]:
# after the progress bar ends, it may still take a while. just a heads up, sorry
df = compute_neighborhoods(
    S=S,
    V=V,
    s_coords=s_coords,
    v_coords=v_coords,
    gene_cols=gene_cols,
    radius=radius
)

df = df.join(S[["cell_type", "sample"]])         # maintaining cell type and sample info for downstream

Processing:   0%|          | 0/88019 [00:00<?, ?it/s]C:\Users\saliu\AppData\Local\Temp\ipykernel_7592\4127043255.py:17: RuntimeWarning: invalid value encountered in scalar divide
  prop_dying = n_dying / tumor                                    # will return some NaN but removed downstream
C:\Users\saliu\AppData\Local\Temp\ipykernel_7592\4127043255.py:20: RuntimeWarning: invalid value encountered in scalar divide
  eff_disc = n_dying / n_immune
C:\Users\saliu\AppData\Local\Temp\ipykernel_7592\4127043255.py:19: RuntimeWarning: invalid value encountered in scalar divide
  eff_cont = prop_dying/ n_immune                                 # dying per cd8, ie. are some T-cells better at killing?
Processing:   3%|▎         | 2922/88019 [00:00<00:02, 29215.92it/s]C:\Users\saliu\AppData\Local\Temp\ipykernel_7592\4127043255.py:19: RuntimeWarning: divide by zero encountered in scalar divide
  eff_cont = prop_dying/ n_immune                                 # dying per cd8, ie. are some T-cells bett

In [13]:
df_clean = df.dropna()                           # removing for NaN efficacy denominator

In [20]:
df_eff = df_clean[df_clean["n_immune"] != 0.0]

In [ ]:
# helps restore independence by sampling non-overlapping neighborhoods using a greedy algorithm.
def non_overlapping(df, n, radius):
    rng = np.random.default_rng(42)              # run reseeding.

    coords = df[['cx', 'cy']].to_numpy()
    remaining_idx = np.arange(len(coords))

    selected_idx = []

    while len(selected_idx) < n and len(remaining_idx) > 0:
        # pick a random remaining point
        pick_i = rng.choice(remaining_idx)
        selected_idx.append(pick_i)

        # build tree of remaining points
        tree = cKDTree(coords[remaining_idx])

        # find all points within radius of the chosen point
        neighbors = tree.query_ball_point(coords[pick_i], r=radius)

        # map neighbor indices back to global indices
        to_remove = set(remaining_idx[neighbors])

        # keep only points not removed
        remaining_idx = np.array([i for i in remaining_idx if i not in to_remove])

    return df.iloc[selected_idx].copy()

In [ ]:
df_sub = non_overlapping(df_clean, n = 1000,                        # for now for comparability with pd1. still >383 so we good. total is 1366 obs
                                       radius=radius*2, seed=42)    # the answer to the ultimate question of life, the universe, and everything.

In [16]:
# verify output
print(df_sub.iloc[300:305, ])
df_sub.shape

             cx        cy  n_alive  n_dying  n_immune  n_lectin  tumor   all  \
34956  4675.780  3899.050     12.0      4.0       2.0       1.0   16.0  19.0   
76388  3739.936  3775.766     13.0     10.0       3.0       0.0   23.0  26.0   
5273   3763.313  2455.760      3.0      0.0       9.0      15.0    3.0  27.0   
80710  4766.045  1874.682     18.0      2.0       3.0       0.0   20.0  23.0   
52955  5924.814  4910.773     30.0      1.0       0.0       0.0   31.0  31.0   

       prop_dying  exist_dying  eff_cont  eff_disc             1  \
34956    0.250000          1.0  0.125000  2.000000  2.747497e-06   
76388    0.434783          1.0  0.144928  3.333333  3.323175e-07   
5273     0.000000          0.0  0.000000  0.000000  9.919731e-08   
80710    0.100000          1.0  0.033333  0.666667  9.523157e-07   
52955    0.032258          1.0       inf       inf  5.498043e-06   

                  2             3             4             5 cell_type sample  
34956  1.119065e-06  1.469424

(1366, 19)

In [20]:
df_sub.to_csv("0609_NSF5_40pd1.csv", index=False)